# H-N140 — H-N00 Dice+CE noSmooth

기본 H-N00에서 `loss.smooth`만 `0.00001 → 0.0`으로 바꾼 단일 변수 실험입니다. 네트워크, Dice+CE 비율, SGD, LR, batch, patch, sampling, epoch 및 early stopping은 H-N00과 동일합니다. 평가를 위해 매 epoch checkpoint, 5 epoch full-volume validation, Top-3 ±4 epoch 재검증을 적용합니다.

In [ ]:
from pathlib import Path
import json,subprocess,time
import pandas as pd
from IPython.display import clear_output,display
ROOT=Path("/home/stu03/workspace/model");DATA_ROOT=Path("/home/stu03/workspace/datasets/hemorrhage/v1/model_ready/nnunet");RESULTS_ROOT=ROOT/"runs/nnunet";RUNNER=ROOT/"scripts/training/run_h_n140.py"
FOLD_DIR=RESULTS_ROOT/"Dataset001_ICHBinary/nnUNetTrainerICHBaseline__ICHPlansBaselineDiceCENoSmoothV1__3d_fullres/fold_0";PROGRESS_FILE=FOLD_DIR/"progress.json";RUN_DIR=ROOT/"runs/batches/H-N140";LOG_FILE=RUN_DIR/"train.log";PID_FILE=RUN_DIR/"train.pid"
print(RUNNER.exists(),RUNNER);print(FOLD_DIR)

## 1. H-N00과 설정 차이 확인

In [ ]:
import yaml
base=yaml.safe_load((ROOT/"configs/experiments/hemorrhage/H-N00.yaml").read_text());trial=yaml.safe_load((ROOT/"configs/experiments/hemorrhage/H-N140.yaml").read_text())
print("H-N00 smooth:",base["loss"]["smooth"]);print("H-N140 smooth:",trial["loss"]["smooth"]);print("H-N140 loss:",trial["loss"])

## 2. GPU 및 dry-run

In [ ]:
subprocess.run(["nvidia-smi"],check=False)
command=[str(ROOT/".venv/bin/python"),str(RUNNER),"--data-root",str(DATA_ROOT),"--results-root",str(RESULTS_ROOT),"--gpu","0"]
subprocess.run(command+["--dry-run"],cwd=ROOT,check=True)

## 3. 백그라운드 학습 시작/자동 재개

In [ ]:
def pid_is_running(path):
 if not path.is_file():return False
 try:
  pid=path.read_text().strip();stat=subprocess.run(["ps","-o","stat=","-p",pid],capture_output=True,text=True).stdout.strip();return bool(stat) and not stat.startswith("Z")
 except (ValueError,OSError):return False
RUN_DIR.mkdir(parents=True,exist_ok=True)
if pid_is_running(PID_FILE):print("이미 실행 중 PID:",PID_FILE.read_text().strip())
else:
 log=LOG_FILE.open("a",encoding="utf-8");process=subprocess.Popen(command,cwd=ROOT,stdout=log,stderr=subprocess.STDOUT,start_new_session=True);PID_FILE.write_text(str(process.pid));print("시작 PID:",process.pid)

## 4. 실시간 진행률

In [ ]:
def read_json(path):
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}

def fmt(seconds):
    if seconds is None:
        return "-"
    seconds = int(seconds)
    hours, remainder = divmod(seconds, 3600)
    minutes, seconds = divmod(remainder, 60)
    return f"{hours:02d}:{minutes:02d}:{seconds:02d}"

def pid_is_running(path):
    if not path.is_file():
        return False
    try:
        pid = path.read_text(encoding="utf-8").strip()
        stat = subprocess.run(
            ["ps", "-o", "stat=", "-p", pid],
            capture_output=True, text=True,
        ).stdout.strip()
        return bool(stat) and not stat.startswith("Z")
    except (ValueError, OSError):
        return False

BEST_FULL_FILE = FOLD_DIR / "best_full_macro.json"

while True:
    clear_output(wait=True)
    progress = read_json(PROGRESS_FILE)
    running = pid_is_running(PID_FILE)
    training_status = progress.get("status")

    if running and training_status == "completed":
        phase = "top3_neighborhood_validation"
        display_status = "running"
    elif running:
        phase = "training"
        display_status = training_status or "starting"
    elif BEST_FULL_FILE.is_file():
        phase = "completed"
        display_status = "completed"
    elif progress:
        phase = "stopped_or_failed"
        display_status = training_status or "stopped"
    else:
        phase = "waiting"
        display_status = "waiting"

    row = {
        "실험": "H-N140",
        "상태": display_status,
        "Phase": phase,
        "Epoch": f"{progress.get('current_epoch', 0)}/{progress.get('total_epochs', '-')}",
        "진행률(%)": progress.get("progress_percent", 0),
        "ETA": fmt(progress.get("eta_seconds")),
        "Train Loss": progress.get("train_loss"),
        "Dice": progress.get("pseudo_dice"),
        "Recall": progress.get("pseudo_recall"),
        "Composite": progress.get("composite"),
        "Best Dice": progress.get("best_dice"),
        "Best Recall": progress.get("best_recall"),
        "Best Composite": progress.get("best_composite"),
        "Full Macro Dice": progress.get("full_macro_dice"),
        "Best Full Macro Dice": progress.get("best_full_macro_dice"),
        "Best Epoch": progress.get("best_full_macro_epoch"),
        "Top-3": progress.get("top_full_macro_epochs"),
        "보존 checkpoint": progress.get("retained_epoch_checkpoints"),
        "GPU(GB)": progress.get("gpu_max_memory_gb"),
        "PID 실행 중": running,
    }
    display(pd.DataFrame([row]))

    if LOG_FILE.is_file():
        print("로그 마지막 10줄:")
        print("".join(LOG_FILE.read_text(encoding="utf-8", errors="replace").splitlines(True)[-10:]))

    if not running and progress:
        break
    time.sleep(10)


## 5. 완료 결과

In [ ]:
print("output:",FOLD_DIR)
for name in ["checkpoint_best_composite.pth","checkpoint_final.pth","validation/summary.json"]:print(name,(FOLD_DIR/name).exists())